# Reproducing PathwayAE (simplified, self-contained, TCGA + METABRIC)
Builds and trains one PAAE model on TCGA-BRCA and evaluates it on METABRIC, matching the paper's approach of using only genes shared between the two datasets.


**Known differences from the paper**:
- Gene expression: STAR-FPKM pipeline instead of the paper's HTSeq-FPKM (Xena has retired the HTSeq version for this cohort)
- METABRIC gene expression: `data_mrna_illumina_microarray.txt` instead of the paper's `data_mrna_agilent_microarray.txt.gz` (the Agilent file is no longer available in the current METABRIC download)
- Pathway definitions: current MSigDB release (v2026.1), renamed to match `v7.5.1` filenames, rather than the exact archived v7.5.1 files
- Single train/test split, single classifier (Logistic Regression), fixed architecture, 100 epochs instead of 1024 — no grid search or cross-validated repeats
- No PAVAE, survival analysis, feature importance/ANPW, or clustering

## Sources

- Gene expression (TCGA): https://xenabrowser.net/datapages/?cohort=GDC%20TCGA%20Breast%20Cancer%20(BRCA) — STAR-FPKM dataset, renamed from `TCGA-BRCA.star_fpkm.tsv` to `TCGA-BRCA.htseq_fpkm.tsv`, placed in `~/data/pathwayae/tcga/`

- Clinical/phenotype (TCGA): `tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/BRCA_clinicalMatrix`, renamed from `BRCA_clinicalMatrix` to `TCGA.BRCA.sampleMap_BRCA_clinicalMatrix`, placed in `~/data/pathwayae/tcga/`

- `ensembl_to_gene_id.json`: copied from the cloned GitHub repo's `data/tcga/` folder, name left unchanged, placed in `~/data/pathwayae/tcga/`

- KEGG pathways: gsea-msigdb.org (after free registration) — go to Molecular Signatures Database, Human Collections, Hallmark Gene Sets, KEGG_LEGACY subset of CP, click on JSON bundle. https://gsea-msigdb.org/gsea/msigdb/human/collections.jsp#H — renamed from `c2.cp.kegg_legacyv2026.1.Hs.txt` to `c2.cp.kegg.v7.5.1.json`, placed in `~/data/pathways/`

- Hallmark pathways: gsea-msigdb.org — go to Molecular Signatures Database, Human Collections, Hallmark Gene Sets, click on JSON bundle. https://gsea-msigdb.org/gsea/msigdb/human/collections.jsp#H — renamed from `h.all.v2026.1.Hs.txt` to `h.all.v7.5.1.json`, placed in `~/data/pathways/`

- METABRIC gene expression: `datahub.s3.amazonaws.com/brca_metabric.tar.gz`, downloaded as `data_mrna_illumina_microarray.txt`, name left unchanged, placed in `~/data/metabric/`

- METABRIC clinical: same source as METABRIC gene expression, `data_clinical_patient.txt`, name left unchanged, placed in `~/data/metabric/`


## 1. Imports

In [ ]:
import os
import json
import functools

import numpy as np
import pandas as pd

import sklearn.model_selection
import sklearn.pipeline
import sklearn.preprocessing
import sklearn.linear_model
import sklearn.metrics

import torch
import torch.nn as nn

import skorch

#from pathwayae.models import PAAE
#from pathwayae.skorch_utils import ScoredNeuralNetAutoencoder
#from pathwayae.losses import AE_MSELoss
#from pathwayae.utils import sample_wise_preprocess_fn
#from pathwayae.pathway_utils import read_pathway_from_json_file

import random

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # no-op if you're on CPU, harmless either way

## 2. Definitions (copied in directly so the notebook has no dependency on the `pathwayae` package)

### MLP / PAAE

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#from .common import *
from typing import Callable

class MLP(nn.Module):
    def __init__(
            self,
            input_dim:int,
            hidden_dims:list[int],
            output_dim:int,
            nonlinearity:Callable,
            dropout_rate:float=0.5,
            bias:bool=True,
            ):
        super().__init__()
        in_dims = [input_dim] + hidden_dims
        out_dims = hidden_dims + [output_dim]
        
        self.layers = nn.ModuleList([nn.Linear(d_in, d_out, bias=bias) for d_in, d_out in zip(in_dims, out_dims)])
        self.nonlinearity = nonlinearity
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x:torch.Tensor) -> torch.Tensor:
        for layer in self.layers[:-1]:
            x = self.dropout(self.nonlinearity(layer(x)))
        return self.layers[-1](x)
    
    def layer_activations(self, x:torch.Tensor) -> list[torch.Tensor]:
        # To allow for activation normalisation
        activations = [x]
        for layer in self.layers[:-1]:
            activations.append(self.dropout(self.nonlinearity(layer(activations[-1]))))
        return activations[1:] + [self.layers[-1](activations[-1])]
    
class PAAE(nn.Module):
    def __init__(
            self,
            genes_dim:int,
            pathway_definitions:list[torch.LongTensor],
            hidden_dims:list[int]=[128],
            encoding_dim:int=64,
            nonlinearity=F.relu,
            dropout_rate:float=0.5,
            bias:bool=True,
            final_nonlinearity=lambda x:x,
            pathway_hidden_dims:list[int] = [],
            pathway_nonlinearities=F.relu,
            pathway_activity_nonlinearity=torch.tanh,
            pathway_dropout_rate:int=0,
            pathway_bias:bool=True,
            ):
        super().__init__()
        if genes_dim is None:
            raise ValueError("Must specify genes dimension before initialising the model")
        try:
            len(hidden_dims)
        except TypeError:
            hidden_dims = [hidden_dims]

        input_dim = len(pathway_definitions)
        
        self.pathway_definitions = pathway_definitions

        encoder_list = [
            MLP(p.shape[0], pathway_hidden_dims, 1, pathway_nonlinearities, pathway_dropout_rate, pathway_bias) for p in pathway_definitions
        ]

        self.pathway_encoders = nn.ModuleList(encoder_list)
        self.encoder = MLP(input_dim, hidden_dims, encoding_dim, nonlinearity, dropout_rate, bias)
        self.decoder = MLP(encoding_dim, hidden_dims[-1::-1], genes_dim, nonlinearity, dropout_rate, bias)
        self.final_nonlinearity = final_nonlinearity
        self.pathway_activity_nonlinearity = pathway_activity_nonlinearity
    
    def get_pathway_activities(self,x:torch.Tensor) -> torch.Tensor:
        return torch.concat(
            [
                self.pathway_activity_nonlinearity(enc(x[:,pway]))
                for pway, enc in
                    zip(self.pathway_definitions, self.pathway_encoders)
            ],
            dim=-1,
        )
    
    def encode_pathways(self,a:torch.Tensor) -> torch.Tensor:
        return self.encoder(a)

    def encode(self, x:torch.Tensor) -> torch.Tensor:
        a = self.get_pathway_activities(x)
        z = self.encode_pathways(a)
        return z
    
    def decode(self,z:torch.Tensor) -> torch.Tensor:
        return self.final_nonlinearity(self.decoder(z))
    
    def forward(self, x:torch.Tensor) -> torch.Tensor:
        z = self.encode(x)
        x_hat = self.decode(z)
        return z, x_hat
    
    def layer_activations(self,x:torch.Tensor) -> list[torch.Tensor]:
        raise NotImplementedError()
    
    def get_feature_importance_matrix(self) -> torch.Tensor:
        raise NotImplementedError()

class PAAE_parallel(PAAE):
    def get_pathway_activities(self,x:torch.Tensor) -> torch.Tensor:
        futures = [
            torch.jit.fork(lambda enc, pway: self.pathway_activity_nonlinearity(enc(x[:,pway])), enc, pway)
            for pway, enc in
                zip(self.pathway_definitions, self.pathway_encoders)
        ]
        return self.pathway_activity_nonlinearity(
            torch.concat(
                [
                    torch.jit.wait(f)
                    for f in futures
                ],
                dim=-1,
            )
        )

### ScoredNeuralNetAutoencoder

In [22]:
class ScoredNeuralNetAutoencoder(skorch.NeuralNet):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def score(self, X, y=None, sample_weight=None):
        return -skorch.scoring.loss_scoring(self, X, X, sample_weight) # Negative so that sklearn gets the smallest loss
    
    def transform(self, X):
        return self.predict(X)

    def full_transform(self, X):
        nonlin = self._get_predict_nonlinearity()
        ys:list[list[np.ndarray]] = None
        for yp in self.forward_iter(X, training=False):
            if not isinstance(yp, tuple):
                yp = (yp,)
            if ys is None:
                ys = [[] for _ in yp]
            for i, _ in enumerate(yp):
                ys[i].append(skorch.utils.to_numpy(nonlin(yp[i])))
        ret = [np.concatenate(ys[i], 0) for i, _ in enumerate(yp)]
        return tuple(ret)

    
    def fit_transform(self, X, y=None, **kwargs):
        self.fit(X, y, **kwargs)
        Xt = self.transform(X)
        return Xt
    
    def fit_loop(self, X, y=None, epochs=None, **fit_params):
        super().fit_loop.__doc__
        return super().fit_loop(X, y=X, epochs=epochs, **fit_params)


### AE_MSELoss

In [23]:
from torch.nn.modules.loss import _Loss as Loss, MSELoss
import typing
from torch import Tensor

AE_Learned_Representation = Tensor
AE_Reconstruction = Tensor

AEOutput = tuple[AE_Learned_Representation, AE_Reconstruction]
class AE_MSELoss(MSELoss):
    def forward(self, output:AEOutput, target:torch.Tensor) -> torch.Tensor:
        _, x_hat = output
        return F.mse_loss(x_hat, target, reduction=self.reduction)

### sample_wise_preprocess_fn

In [24]:
def sample_wise_preprocess_fn(
        X:pd.DataFrame|np.ndarray,
        input_is_log2pk:bool=True, input_k:float=1, input_type:str="fpkm",
        output_is_log2pk:bool=True, output_k:float=1, output_type:str="tpm",
        copy:bool=True, working_precision=np.longdouble,
        lower_clip_eps:float|None=2e-6,
        assert_finite = True,
        ) -> np.ndarray:
    """
    Helper function to transfer between normalization types using only sample information.
    This function should be safe to use before a machine learning pipeline since it should only consider information within each sample.
    By default it works with float128 precision due to some of the rounding losses due to some of the large scales involved, if one is getting out of memory errors change the precision to a lower precision.
    """
    x = X.copy() if copy else X
    x = x.astype(working_precision)

    if input_is_log2pk and input_type!=output_type:
        x = from_log2pk(x, k=input_k)
    
    if input_type!=output_type:
        if input_type in {"fpkm", "fpks", "fpku"} and output_type in {"fpkm", "fpks", "fpku", "tpm", "tps", "tpu"}:
            x = x * 1e6 if input_type[-1] == "u" else x
            x = x * (1e6/np.sqrt(x.shape[1])) if input_type[-1] == "s" else x
            x = fpkm_to_tpm(x, gene_axis=1) if output_type[:-1] == "tp" else x
            x = x/1e6 if output_type[-1] == "u" else x
            x = x*(np.sqrt(x.shape[1])/1e6) if output_type[-1] == "s" else x
        else:
            raise NotImplementedError(f"Conversion from {input_type} to {output_type} not implemented.")
    
    if output_is_log2pk and ( 
            (input_is_log2pk and input_type!=output_type)
            or
            (not input_is_log2pk)
        ):
        x = to_log2pk(x, k=output_k)

    if lower_clip_eps is not None:
        x = np.where(x<=lower_clip_eps,0,x)
    
    assert (not assert_finite) or np.isfinite(x).all(), ValueError("Some values in the array are non-finite")

    return x

### read_pathway_from_json_file

In [25]:
import functools
import json

def read_pathway_from_json_file(pathway_description_path:str, allowed_genes:set=None):
    with open(pathway_description_path, "r") as f:
        pathway_descriptions = json.load(f)
    pathway_genes = [pathway_descriptions[k]["geneSymbols"] for k in pathway_descriptions]
    if allowed_genes is None:
        return pathway_genes
    else:
        all_pathway_genes = functools.reduce(lambda acc, v: acc.union(set(v)), pathway_genes, set())
        common_genes = all_pathway_genes.intersection(allowed_genes)
        pathway_genes_with_allowed_genes = [[gene for gene in pathway if gene in common_genes] for pathway in pathway_genes]
        return pathway_genes_with_allowed_genes

## 3. Paths

In [26]:
data_folder = os.path.expanduser("~/data/")
tcga_folder = os.path.join(data_folder, "pathwayae", "tcga")
pathway_folder = os.path.join(data_folder, "pathways")
meta_folder = os.path.join(data_folder, "metabric")
os.makedirs(meta_folder, exist_ok=True)

cancer_type = "BRCA"


## 4. Load TCGA gene expression

In [27]:
gex = pd.read_csv(
    os.path.join(tcga_folder, f"TCGA-{cancer_type}.htseq_fpkm.tsv"),
    sep="\t", index_col="Ensembl_ID",
).T.dropna(axis="columns")

# Strip version suffixes so the mapping works regardless of GENCODE version differences
gex.columns = gex.columns.str.split(".").str[0]

with open(os.path.join(tcga_folder, "ensembl_to_gene_id.json")) as f:
    ensembl_to_gex_dict = json.load(f)
ensembl_to_gex_dict = {k.split(".")[0]: v for k, v in ensembl_to_gex_dict.items()}

columns_to_drop = [k for k in ensembl_to_gex_dict if ensembl_to_gex_dict[k] == ""]
gex = gex.drop(columns=columns_to_drop, errors="ignore")
gex = gex.rename(columns=ensembl_to_gex_dict)

gex.columns.rename("GeneName", inplace=True)
gex.index.rename("SampleID", inplace=True)

gex.shape


(1226, 60660)

## 5. Load TCGA clinical labels (subtype)

In [28]:
ensembl_fpkm_phenotype_tsv_fname = os.path.join(
    tcga_folder, f"TCGA.{cancer_type}.sampleMap_{cancer_type}_clinicalMatrix"
)
phenotype = pd.read_csv(ensembl_fpkm_phenotype_tsv_fname, sep="\t", index_col="sampleID")
phenotype.index = phenotype.index.rename("SampleID")
phenotype = phenotype[[c for c in phenotype.columns if "pam50" in c.lower()]]

phenotype_clf_tgt = "PAM50Call_RNAseq"
phenotype_clf_map = {"LumA": 0, "LumB": 1, "Basal": 2, "Normal": 3, "Her2": 4}
phenotype_clf_nan = {f"{v}": np.nan for v in [np.nan, "not reported", ""]}

PHENOTYPE_CLF_COLUMN = "subtype"
phenotype[PHENOTYPE_CLF_COLUMN] = phenotype[phenotype_clf_tgt].replace(phenotype_clf_nan)
phenotype = phenotype.dropna(subset=[PHENOTYPE_CLF_COLUMN])

# Line up phenotype sample IDs with gex sample IDs (TCGA IDs don't match exactly by default)
_possible_mappings = {idx: [] for idx in phenotype.index}
for idx in phenotype.index:
    for v in gex[gex.index.str.startswith(idx)].index.values:
        _possible_mappings[idx].append(v)
_replacements = {k: sorted(v)[0] for k, v in _possible_mappings.items() if len(v) > 0}
phenotype = phenotype.rename(index=_replacements, inplace=False)

both_index = sorted(set(phenotype.index).intersection(gex.index))
gex = gex.loc[both_index]
phenotype = phenotype.loc[both_index, [PHENOTYPE_CLF_COLUMN]]
phenotype[PHENOTYPE_CLF_COLUMN] = phenotype[PHENOTYPE_CLF_COLUMN].replace(phenotype_clf_map)

assert all(gi == pi for gi, pi in zip(gex.index.to_list(), phenotype.index.to_list()))

label_encoder = sklearn.preprocessing.LabelEncoder()
y = label_encoder.fit_transform(phenotype[PHENOTYPE_CLF_COLUMN])

gex.shape, y.shape


C:\Users\micha\AppData\Local\Temp\ipykernel_18096\1281406189.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  phenotype[PHENOTYPE_CLF_COLUMN] = phenotype[PHENOTYPE_CLF_COLUMN].replace(phenotype_clf_map)


((953, 60660), (953,))

## 6. Load METABRIC gene expression

In [29]:
metabric_gex_path = os.path.join(meta_folder, "data_mrna_illumina_microarray.txt")
metabric_clinical_path = os.path.join(meta_folder, "data_clinical_patient.txt")

gex_meta = pd.read_csv(
    metabric_gex_path, sep="\t", index_col="Hugo_Symbol"
).drop(columns="Entrez_Gene_Id")
gex_meta.index.rename("SampleID", inplace=True)
gex_meta = gex_meta.T.dropna(axis="columns")
gex_meta.columns = gex_meta.columns.str.upper()

gex_meta.shape


(1980, 20592)

## 7. Load METABRIC clinical labels

In [30]:
metabric_phenotype = pd.read_csv(
    metabric_clinical_path, sep="\t", comment="#", index_col="PATIENT_ID"
)
metabric_phenotype.index.rename("SampleID", inplace=True)
metabric_phenotype = metabric_phenotype[["CLAUDIN_SUBTYPE"]].copy()
metabric_phenotype.rename(columns={"CLAUDIN_SUBTYPE": "PAM50"}, inplace=True)
metabric_phenotype.dropna(inplace=True)
metabric_phenotype = metabric_phenotype[~metabric_phenotype["PAM50"].isin(["claudin-low", "NC"])]

both_index_meta = sorted(set(metabric_phenotype.index).intersection(gex_meta.index))
gex_meta = gex_meta.loc[both_index_meta]
metabric_phenotype = metabric_phenotype.loc[both_index_meta]

assert all(gex_meta.index == metabric_phenotype.index)
gex_meta.shape, metabric_phenotype.shape


((1756, 20592), (1756, 1))

## 8. Intersect genes between TCGA and METABRIC
**Key fix:** this now happens *before* the model is built, and restricts `gex` (TCGA) as well as `gex_meta`, not just `gex_meta`. The paper does this same intersection as a preprocessing step (Section II.A: "those that appear on both datasets were taken, resulting in dx = 18593 features per sample"). Doing it after training (as in the previous version) fails, since the trained model's input size is fixed at training time.

In [31]:
gex = gex.loc[:, ~gex.columns.duplicated()]
gex_meta = gex_meta.loc[:, ~gex_meta.columns.duplicated()]

genes_in_both = sorted(set(gex.columns).intersection(set(gex_meta.columns)))
print("Number of common genes:", len(genes_in_both))

gex = gex[genes_in_both]
gex_meta = gex_meta.loc[:, genes_in_both]

gex.shape, gex_meta.shape


Number of common genes: 18807


((953, 18807), (1756, 18807))

## 9. Convert METABRIC labels to the same numeric encoding as TCGA

In [32]:
y_metabric = metabric_phenotype["PAM50"].replace(phenotype_clf_map)
assert not y_metabric.isna().any()
y_metabric = y_metabric.astype(int)

X_metabric = gex_meta.values.astype(np.float32)
assert np.isfinite(X_metabric).all()


C:\Users\micha\AppData\Local\Temp\ipykernel_18096\3673719601.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_metabric = metabric_phenotype["PAM50"].replace(phenotype_clf_map)


## 10. Load pathway definitions
Runs after the TCGA/METABRIC gene intersection, so pathway indices are computed against the shared gene set both datasets can use.

In [33]:
gex_genes = set(gex.columns.values)
gex_genes_indexer = {v: i for i, v in enumerate(gex.columns.values)}
get_pathways_with_indices = lambda pathways: [
    [gex_genes_indexer[gene] for gene in pathway] for pathway in pathways
]

kegg_pathways = read_pathway_from_json_file(
    os.path.join(pathway_folder, "c2.cp.kegg.v7.5.1.json"), gex_genes
)
kegg_pathways_with_indices = get_pathways_with_indices(kegg_pathways)

hallmark_pathways = read_pathway_from_json_file(
    os.path.join(pathway_folder, "h.all.v7.5.1.json"), gex_genes
)
hallmark_pathways_with_indices = get_pathways_with_indices(hallmark_pathways)

len(kegg_pathways_with_indices), len(hallmark_pathways_with_indices)


(186, 50)

## 11. Build ONE PAAE model
`genes_dim` now reflects the shared TCGA/METABRIC gene count, not the full TCGA-only gene set.

In [34]:

genes_dim = gex.values.shape[1]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

MAX_EPOCHS = 100   # paper used 1024 — lowered here for a quick test run, raise it back for a fuller reproduction
DEFAULT_LR = 1e-4

hallmark_p = [torch.tensor(pathway) for pathway in hallmark_pathways_with_indices]

paae_pipeline = sklearn.pipeline.Pipeline([
    ("scale", sklearn.preprocessing.QuantileTransformer(
        n_quantiles=gex.values.shape[0], output_distribution="normal",
    )),
    ("net", ScoredNeuralNetAutoencoder(
        PAAE,
        module__genes_dim=genes_dim,
        module__pathway_definitions=hallmark_p,
        module__hidden_dims=[128],
        module__pathway_hidden_dims=[128],
        max_epochs=MAX_EPOCHS,
        lr=DEFAULT_LR,
        iterator_train__shuffle=True,
        criterion=AE_MSELoss,
        optimizer=torch.optim.Adam,
        device=device,
    )),
    ("clf", sklearn.linear_model.LogisticRegression(max_iter=1000)),
])


Using device: cpu


## 12. Train on TCGA and check internal test accuracy

In [35]:
X = gex.values.astype(np.float32)
X = sample_wise_preprocess_fn(
    X, copy=False,
    input_is_log2pk=True, input_k=1, input_type="fpkm",
    output_is_log2pk=True, output_k=1e-3, output_type="fpkm",
)
X = X.astype(np.float32)  # sample_wise_preprocess_fn upcasts internally (Windows longdouble fix) - cast back down

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

paae_pipeline.fit(X_train, y_train)

y_pred = paae_pipeline.predict(X_test)
print("Test accuracy:", sklearn.metrics.accuracy_score(y_test, y_pred))


c:\Users\micha\anaconda3\envs\pathwayae\lib\site-packages\sklearn\preprocessing\_data.py:2846: UserWarning: n_quantiles (953) is greater than the total number of samples (762). n_quantiles is set to n_samples.
  warnings.warn(


  epoch    train_loss    valid_loss     dur
-------  ------------  ------------  ------
      1        3.9179        3.8877  0.5236
      2        3.9146        3.8847  0.3766
      3        3.9111        3.8815  0.3456
      4        3.9077        3.8777  0.3532
      5        3.9035        3.8729  0.3766
      6        3.8982        3.8665  0.3951
      7        3.8908        3.8576  0.3838
      8        3.8810        3.8451  0.3809
      9        3.8681        3.8278  0.4018
     10        3.8515        3.8041  0.4708
     11        3.8285        3.7723  0.4186
     12        3.7979        3.7301  0.5423
     13        3.7586        3.6754  0.4285
     14        3.7101        3.6053  0.4364
     15        3.6487        3.5175  0.4359
     16        3.5827        3.4102  0.4525
     17        3.4990        3.2826  0.4397
     18        3.4102        3.1386  0.5072
     19        3.3148        2.9861  0.4301
     20        3.2240        2.8354  0.4364
     21        3.1429        2.7

## 13. Apply the trained model to METABRIC (external validation)

In [36]:
metabric_predictions = paae_pipeline.predict(X_metabric)
print(metabric_predictions[:10])


[0 3 0 3 0 0 3 0 0 0]


## 14. Evaluate external validation performance

In [37]:
metabric_accuracy = sklearn.metrics.accuracy_score(y_metabric, metabric_predictions)
metabric_precision = sklearn.metrics.precision_score(y_metabric, metabric_predictions, average="macro")
metabric_recall = sklearn.metrics.recall_score(y_metabric, metabric_predictions, average="macro")
metabric_f1 = sklearn.metrics.f1_score(y_metabric, metabric_predictions, average="macro")

print("METABRIC External Validation")
print("----------------------------")
print(f"Accuracy:  {metabric_accuracy:.4f}")
print(f"Precision: {metabric_precision:.4f}")
print(f"Recall:    {metabric_recall:.4f}")
print(f"F1 Score:  {metabric_f1:.4f}")


METABRIC External Validation
----------------------------
Accuracy:  0.3331
Precision: 0.1832
Recall:    0.2849
F1 Score:  0.1652


c:\Users\micha\anaconda3\envs\pathwayae\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 15. ROC AUC on METABRIC

In [38]:
metabric_probabilities = paae_pipeline.predict_proba(X_metabric)
metabric_auc = sklearn.metrics.roc_auc_score(
    y_metabric, metabric_probabilities, multi_class="ovr", average="macro"
)
print(f"ROC AUC: {metabric_auc:.4f}")


ROC AUC: 0.7856
